# Audio Head Ablation Test

This notebook tests a simple audio forward pass, cache inspection, and a head ablation hook using the forked TransformerLens repo.

In [ ]:
!pip -q install -U pip setuptools wheel
!git clone https://github.com/david-wei-01001/TransformerLens.git
%cd TransformerLens
!pip -q install -e .
print("\n⚠️ IMPORTANT: Restart runtime now, then run the next cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Cloning into 'TransformerLens'...
remote: Enumerating objects: 5328, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 5328 (delta 126), reused 78 (delta 67), pack-reused 5138 (from 2)
Receiving objects: 100% (5328/5328), 25.07 MiB | 19.58 MiB/s, done.
Resolving deltas: 100% (3580/3580), done.
/content/TransformerLens
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing b

In [1]:
import math
from typing import Optional, Tuple

import numpy as np
import torch

import transformer_lens.utils as utils
from transformer_lens import HookedAudioEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_RATE = 16000
DURATION_S = 1.0
torch.set_grad_enabled(False)
print('Device:', DEVICE)

Device: cuda


In [2]:
def make_sine(
    sr: int = SAMPLE_RATE,
    duration: float = DURATION_S,
    freq: float = 440.0,
    amp: float = 0.1,
) -> np.ndarray:
    t = np.linspace(0, duration, int(sr * duration), endpoint=False, dtype=np.float32)
    return amp * np.sin(2 * math.pi * freq * t)


def get_output_repr(
    model: HookedAudioEncoder,
    frames: torch.Tensor,
    frame_mask: Optional[torch.Tensor],
    hooks=None,
) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[dict]]:
    """Return a pooled representation plus optional logits/cache."""
    if hooks is None:
        try:
            out = model(frames, one_zero_attention_mask=frame_mask)
        except TypeError:
            out = model(frames)
    else:
        try:
            out = model.run_with_hooks(frames, fwd_hooks=hooks, one_zero_attention_mask=frame_mask)
        except TypeError:
            out = model.run_with_hooks(frames, fwd_hooks=hooks)

    if isinstance(out, torch.Tensor):
        return out.mean(dim=1) if out.ndim >= 2 else out.unsqueeze(0), out, None

    if isinstance(out, dict):
        for key in ('logits', 'ctc_logits', 'predictions'):
            if key in out and isinstance(out[key], torch.Tensor):
                logits = out[key]
                pooled = logits.mean(dim=1) if logits.ndim == 3 else logits
                return pooled, logits, None

    raise RuntimeError(f'Unknown output type from model: {type(out)}')

In [3]:
# Load model
audio_model = HookedAudioEncoder.from_pretrained(
    'facebook/hubert-base-ls960',
    device=DEVICE,
)
audio_model.eval()
print('Loaded model')

If using HuBERT for interpretability research, keep in mind that HuBERT has some significant architectural differences to GPT. For example, LayerNorms are applied *after* the attention and MLP components, meaning that the last LayerNorm in a block cannot be folded.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

Moving model to device:  cuda
Loaded pretrained model facebook/hubert-base-ls960 into HookedEncoder
Loaded model


In [4]:
# Create a one-second sine wave and convert to frames
wav = make_sine()
raw_batch = [wav]

frames, frame_mask = audio_model.to_frames(
    raw_batch,
    sampling_rate=SAMPLE_RATE,
    move_to_device=True,
)

print('frames.shape:', tuple(frames.shape))
if frame_mask is not None:
    print('frame_mask.shape:', tuple(frame_mask.shape))

frames.shape: (1, 49, 768)
frame_mask.shape: (1, 49)


In [5]:
# Inspect one attention pattern from cache
layer_to_visualize = 0

try:
    logits, cache = audio_model.run_with_cache(
        frames,
        one_zero_attention_mask=frame_mask,
        remove_batch_dim=True,
    )
except TypeError:
    logits, cache = audio_model.run_with_cache(frames, remove_batch_dim=True)

pattern_name = utils.get_act_name('pattern', layer_to_visualize)
if pattern_name in cache:
    attention_pattern = cache[pattern_name]
else:
    attention_pattern = cache[('pattern', layer_to_visualize, 'attn')]

print('attention_pattern.shape:', tuple(attention_pattern.shape))

attention_pattern.shape: (12, 49, 49)


In [6]:
# Define a simple value ablation hook
layer_to_ablate = 0
head_index_to_ablate = 0
v_act_name = utils.get_act_name('v', layer_to_ablate)

def head_ablation_hook(value: torch.Tensor, hook):
    v = value.clone()
    if v.ndim == 4:       # [batch, pos, heads, d_head]
        v[:, :, head_index_to_ablate, :] = 0.0
    elif v.ndim == 3:     # [pos, heads, d_head]
        v[:, head_index_to_ablate, :] = 0.0
    else:
        raise RuntimeError(f'Unexpected v tensor ndim={v.ndim}')
    return v

print('Hook target:', v_act_name)

Hook target: blocks.0.attn.hook_v


In [7]:
# Baseline vs ablated representations
baseline_repr, baseline_logits, _ = get_output_repr(audio_model, frames, frame_mask, hooks=None)
ablated_repr, ablated_logits, _ = get_output_repr(
    audio_model,
    frames,
    frame_mask,
    hooks=[(v_act_name, head_ablation_hook)],
)

cos = torch.nn.functional.cosine_similarity(baseline_repr, ablated_repr, dim=-1)
print('baseline_repr.shape:', tuple(baseline_repr.shape))
print('ablated_repr.shape:', tuple(ablated_repr.shape))
print('cosine_similarity:', cos.detach().cpu().numpy())

if isinstance(baseline_logits, torch.Tensor) and isinstance(ablated_logits, torch.Tensor):
    b_ids = baseline_logits.argmax(dim=-1)
    a_ids = ablated_logits.argmax(dim=-1)
    print('baseline argmax ids (first 40):', b_ids[0][:40].detach().cpu().tolist())
    print('ablated  argmax ids (first 40):', a_ids[0][:40].detach().cpu().tolist())

baseline_repr.shape: (1, 768)
ablated_repr.shape: (1, 768)
cosine_similarity: [0.71230596]
baseline argmax ids (first 40): [482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482]
ablated  argmax ids (first 40): [482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482]


## Interpretation

- Cosine close to 1: the ablated head likely has little effect.
- Larger drop: the head likely contributes meaningfully for this input.
- If the model returns logits, compare argmax outputs before and after ablation too.